<a href="https://colab.research.google.com/github/ldookria/AI-ML_dildora/blob/main/car_price_Dildora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/Neeraj21439/used-car-price-prediction/main/Car%20details%20v3.csv"
df_raw = pd.read_csv(url)     # never change df_raw. Work on copies.

In [ ]:
print(df_raw.shape)

(8128, 13)


In [ ]:
df_raw.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


In [ ]:
df_raw.describe()

,year,selling_price,km_driven,seats
count,8128.000000,8.128000e+03,8.128000e+03,7907.000000
mean,2013.804011,6.382718e+05,6.981951e+04,5.416719
std,4.044249,8.062534e+05,5.655055e+04,0.959588
min,1983.000000,2.999900e+04,1.000000e+00,2.000000
25%,2011.000000,2.549990e+05,3.500000e+04,5.000000
50%,2015.000000,4.500000e+05,6.000000e+04,5.000000
75%,2017.000000,6.750000e+05,9.800000e+04,5.000000
max,2020.000000,1.000000e+07,2.360457e+06,14.000000


In [ ]:
df_raw.isnull().sum()

,0
name,0
year,0
selling_price,0
km_driven,0
fuel,0
seller_type,0
transmission,0
owner,0
mileage,221
engine,221


In [ ]:
df_raw.duplicated().sum()

np.int64(1202)

In [ ]:
df_raw["owner"].value_counts()

,count
owner,
First Owner,5289
Second Owner,2105
Third Owner,555
Fourth & Above Owner,174
Test Drive Car,5


In [ ]:
df_raw["fuel"].value_counts()

,count
fuel,
Diesel,4402
Petrol,3631
CNG,57
LPG,38


The five biggest problems in the dataset are:

1. There are 1,202 duplicate rows.
2. Some columns, such as mileage, engine, max_power, and torque, contain numbers mixed with text.
3. Several columns contain missing values.
4. The km_driven column contains extreme values, including 2,360,457 kilometers.
5. The torque column contains messy and inconsistent text formats.

In [ ]:
 df = df_raw.copy()

df = df.drop_duplicates().copy()

print(df.shape)

(6926, 13)


In [ ]:
df = df[df["owner"] != "Test Drive Car"].copy()

print(df.shape)

(6921, 13)


In [ ]:
df = df[df["km_driven"] <= 500_000].copy()

print(df.shape)

(6918, 13)


In [ ]:
df = df[df["km_driven"] <= 500_000].copy()

print(df.shape)

(6918, 13)


In [ ]:
def clean(df):
    out = df.copy()

    # Extract the first number from text columns
    for col in ["mileage", "engine", "max_power"]:
        out[col] = pd.to_numeric(
            out[col].astype(str).str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False),
            errors="coerce"
        )

    # Zero horsepower is impossible, so treat it as missing
    out.loc[out["max_power"] == 0, "max_power"] = np.nan

    # Create new columns
    out["brand"] = out["name"].str.split().str[0]
    out["car_age"] = 2020 - out["year"]

    # Remove columns we do not need
    out = out.drop(columns=["name", "year", "torque"])

    return out


df_clean = clean(df)

print(df_clean.shape)

(6918, 12)


In [ ]:
df_clean[["mileage", "engine", "max_power", "seats"]].isnull().sum()

,0
mileage,208
engine,208
max_power,209
seats,208


In [ ]:
brand_counts = df_clean["brand"].value_counts()

rare_brands = brand_counts[brand_counts < 20].index.tolist()

df_clean["brand"] = df_clean["brand"].where(
    ~df_clean["brand"].isin(rare_brands),
    "Other"
)

print("Rare brands:", rare_brands)
print("Number of brands:", df_clean["brand"].nunique())

Rare brands: ['Mitsubishi', 'Volvo', 'Jaguar', 'Isuzu', 'Force', 'Ambassador', 'Daewoo', 'MG', 'Land', 'Kia', 'Lexus', 'Ashok', 'Opel', 'Peugeot']
Number of brands: 19


In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["selling_price"])
y = df_clean["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5534, 11)
X_test: (1384, 11)
y_train: (5534,)
y_test: (1384,)


In [ ]:
print("Missing values in X_train:")
print(X_train.isnull().sum().sum())

print("\nFuel data type:")
print(X_train["fuel"].dtype)

print("\nIs selling_price in X_train?")
print("selling_price" in X_train.columns)

Missing values in X_train:
689

Fuel data type:
object

Is selling_price in X_train?
False


In [ ]:
numeric_cols = [
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats",
    "car_age"
]

onehot_cols = [
    "fuel",
    "seller_type",
    "transmission",
    "brand"
]

ordinal_cols = ["owner"]

owner_order = [
    ["First Owner", "Second Owner", "Third Owner", "Fourth & Above Owner"]
]

print("Numeric columns:", numeric_cols)
print("One-hot columns:", onehot_cols)
print("Ordinal columns:", ordinal_cols)
print("Owner order:", owner_order)

Numeric columns: ['km_driven', 'mileage', 'engine', 'max_power', 'seats', 'car_age']
One-hot columns: ['fuel', 'seller_type', 'transmission', 'brand']
Ordinal columns: ['owner']
Owner order: [['First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner']]


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

onehot_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=owner_order,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

prep = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("onehot", onehot_pipe, onehot_cols),
    ("owner", ordinal_pipe, ordinal_cols)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

def make(model):
    reg = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    return Pipeline([
        ("prep", prep),
        ("model", reg)
    ])

print("Model pipeline function created successfully.")

Model pipeline function created successfully.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

models = {
    "Linear Regression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

print("Models created:")
for name in models:
    print(name)

Models created:
Linear Regression
KNN
Random Forest
Gradient Boosting


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

results_rows = []
fitted_pipes = {}

for name, model in models.items():
    print(f"Training {name}...")

    model_pipe = make(model)

    cv_r2 = cross_val_score(
        model_pipe,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    ).mean()

    model_pipe.fit(X_train, y_train)

    train_predictions = model_pipe.predict(X_train)
    test_predictions = model_pipe.predict(X_test)

    train_mae = mean_absolute_error(y_train, train_predictions)
    test_mae = mean_absolute_error(y_test, test_predictions)
    test_r2 = r2_score(y_test, test_predictions)

    results_rows.append({
        "model": name,
        "cv_r2": cv_r2,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

    fitted_pipes[name] = model_pipe

results = pd.DataFrame(results_rows).sort_values(
    by="test_mae"
).reset_index(drop=True)

display(results)

Training Linear Regression...
Training KNN...
Training Random Forest...
Training Gradient Boosting...


,model,cv_r2,train_mae,test_mae,test_r2
0,Random Forest,0.900516,29958.590075,78776.117154,0.901944
1,Gradient Boosting,0.888488,75714.020337,83150.561857,0.895659
2,KNN,0.840719,73749.812892,95392.837273,0.854633
3,Linear Regression,0.875196,93083.181026,101593.041343,0.825641


In [ ]:
best_model_name = results.iloc[0]["model"]

pipe = fitted_pipes[best_model_name]

print("Best model:", best_model_name)
print("Pipeline saved as: pipe")

Best model: Random Forest
Pipeline saved as: pipe


In [ ]:
print(type(pipe))
print(pipe.predict(X_test.head(3)))
print("Number of predictions:", len(pipe.predict(X_test.head(3))))

<class 'sklearn.pipeline.Pipeline'>
[155262.29307618 695238.77284142 532132.1412276 ]
Number of predictions: 3


In [ ]:
clients = pd.DataFrame([
    ["Maruti Swift VXI", 2017, 0, 40000, "Petrol", "Individual", "Manual", "First Owner", "21.21 kmpl", "1197 CC", "81.80 bhp", "113Nm@ 4200rpm", 5.0],
    ["Hyundai Creta 1.6 SX", 2018, 0, 35000, "Diesel", "Dealer", "Manual", "First Owner", "19.67 kmpl", "1582 CC", "126.2 bhp", "259.9Nm@ 1900-2750rpm", 5.0],
    ["Toyota Innova 2.5 G", 2010, 0, 150000, "Diesel", "Individual", "Manual", "Second Owner", "12.99 kmpl", "2494 CC", "100.6 bhp", "200Nm@ 1200-3600rpm", 8.0],
    ["BMW 5 Series 520d", 2016, 0, 60000, "Diesel", "Dealer", "Automatic", "First Owner", "18.12 kmpl", "1995 CC", "190 bhp", "400Nm@ 1750-2500rpm", 5.0],
    ["Tata Nano Cx", 2012, 0, 70000, "Petrol", "Individual", "Manual", "Second Owner", "23.9 kmpl", "624 CC", "37.48 bhp", "51Nm@ 4000+/-500rpm", 4.0],
], columns=df_raw.columns)

display(clients)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift VXI,2017,0,40000,Petrol,Individual,Manual,First Owner,21.21 kmpl,1197 CC,81.80 bhp,113Nm@ 4200rpm,5.0
1,Hyundai Creta 1.6 SX,2018,0,35000,Diesel,Dealer,Manual,First Owner,19.67 kmpl,1582 CC,126.2 bhp,259.9Nm@ 1900-2750rpm,5.0
2,Toyota Innova 2.5 G,2010,0,150000,Diesel,Individual,Manual,Second Owner,12.99 kmpl,2494 CC,100.6 bhp,200Nm@ 1200-3600rpm,8.0
3,BMW 5 Series 520d,2016,0,60000,Diesel,Dealer,Automatic,First Owner,18.12 kmpl,1995 CC,190 bhp,400Nm@ 1750-2500rpm,5.0
4,Tata Nano Cx,2012,0,70000,Petrol,Individual,Manual,Second Owner,23.9 kmpl,624 CC,37.48 bhp,51Nm@ 4000+/-500rpm,4.0


In [ ]:
clients_clean = clean(clients)

print(clients_clean.shape)
display(clients_clean)

(5, 12)


,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats,brand,car_age
0,0,40000,Petrol,Individual,Manual,First Owner,21.21,1197,81.80,5.0,Maruti,3
1,0,35000,Diesel,Dealer,Manual,First Owner,19.67,1582,126.20,5.0,Hyundai,2
2,0,150000,Diesel,Individual,Manual,Second Owner,12.99,2494,100.60,8.0,Toyota,10
3,0,60000,Diesel,Dealer,Automatic,First Owner,18.12,1995,190.00,5.0,BMW,4
4,0,70000,Petrol,Individual,Manual,Second Owner,23.90,624,37.48,4.0,Tata,8


In [ ]:
clients_clean["brand"] = clients_clean["brand"].where(
    ~clients_clean["brand"].isin(rare_brands),
    "Other"
)

print(clients_clean["brand"])

0     Maruti
1    Hyundai
2     Toyota
3        BMW
4       Tata
Name: brand, dtype: object


In [ ]:
client_X = clients_clean.drop(columns=["selling_price"])

client_predictions = pipe.predict(client_X)

for name, price in zip(clients["name"], client_predictions):
    print(
        f"{name}: ₹{price:,.0f} "
        f"({price / 100_000:.2f} lakh)"
    )

Maruti Swift VXI: ₹554,029 (5.54 lakh)
Hyundai Creta 1.6 SX: ₹1,118,855 (11.19 lakh)
Toyota Innova 2.5 G: ₹481,916 (4.82 lakh)
BMW 5 Series 520d: ₹2,315,435 (23.15 lakh)
Tata Nano Cx: ₹65,285 (0.65 lakh)


In [ ]:
# ===== AUTO-CHECK: paste this cell at the END of your notebook and run it =====
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
_ok = 0; _all = 0
def check(name, cond, hint):
    global _ok, _all
    _all += 1
    try: passed = bool(cond())
    except Exception as e: passed = False; hint = f"{hint}  (error: {e})"
    _ok += passed
    print(("PASS  " if passed else "FAIL  ") + name + ("" if passed else "\n      hint: " + hint))

print("--- Part A-B: cleaning ---")
check("A1 raw data loaded: 8128 rows, 13 columns", lambda: df_raw.shape == (8128, 13), "Load the CSV into df_raw. Don't change df_raw later.")
check("B  clean data has 6918 rows", lambda: len(df_clean) == 6918, "Order: duplicates -> Test Drive Car -> km_driven > 500000. Check each step's row count.")
check("B  no 'Test Drive Car' rows", lambda: "Test Drive Car" not in set(df_clean["owner"]), "Remove rows where owner == 'Test Drive Car'.")
check("B  km_driven outliers removed", lambda: df_clean["km_driven"].max() <= 500_000, "Keep only km_driven <= 500000.")
check("B  mileage, engine, max_power are numbers", lambda: all(pd.api.types.is_numeric_dtype(df_clean[c]) for c in ["mileage","engine","max_power"]), "Take the number out of '23.4 kmpl', '1248 CC', '74 bhp'.")
check("B  no 0 bhp cars", lambda: (df_clean["max_power"] == 0).sum() == 0, "0 bhp is impossible -> make it NaN.")
check("B  name, year, torque removed", lambda: not {"name","year","torque"} & set(df_clean.columns), "Drop these 3 columns after you make brand and car_age.")
check("B  car_age column exists (2020 - year)", lambda: "car_age" in df_clean and df_clean["car_age"].between(0, 40).all(), "car_age = 2020 - year")
check("B  19 brands (rare < 20 cars -> 'Other')", lambda: df_clean["brand"].nunique() == 19 and "Other" in set(df_clean["brand"]), "brand = first word of name. Brands with fewer than 20 cars -> 'Other'.")

print("--- Part C: split + leakage ---")
check("C  train 5534 rows, test 1384 rows", lambda: len(X_train) == 5534 and len(X_test) == 1384, "train_test_split(X, y, test_size=0.2, random_state=42)")
check("C  target is NOT in the features", lambda: "selling_price" not in X_train.columns, "X = df_clean.drop(columns=['selling_price'])  <- LEAK!")
check("C  empty cells NOT filled before split", lambda: X_train.isnull().sum().sum() > 0, "Filling (median) learns from data -> it must be INSIDE the pipeline, not before the split.")
check("C  text NOT encoded before split", lambda: not pd.api.types.is_numeric_dtype(X_train["fuel"]), "Encoding must be INSIDE the pipeline. X_train should still have words.")

print("--- Part D-E: pipeline + models ---")
check("D  pipe is a sklearn Pipeline", lambda: isinstance(pipe, Pipeline), "pipe = Pipeline([('prep', ...), ('model', ...)])")
check("D  pipe works on raw X_test", lambda: len(pipe.predict(X_test.head(3))) == 3, "The pipeline must fill, encode and scale by itself.")
check("E  results table has 3+ models incl. Linear Regression", lambda: len(results) >= 3 and results["model"].str.contains("Linear").any(), "Train at least Linear Regression + 2 other models.")
check("E  results has cv_r2, train_mae, test_mae, test_r2", lambda: {"cv_r2","train_mae","test_mae","test_r2"} <= set(results.columns), "Make a DataFrame with these columns.")
check("E  best model test R2 >= 0.88", lambda: results["test_r2"].max() >= 0.88, "Try Random Forest / Gradient Boosting, and train on log(price).")
check("E  best model test MAE <= 90,000 rupees", lambda: results["test_mae"].min() <= 90_000, "MAE must be in RUPEES. If you trained on log(price), convert back with np.expm1.")

print("--- Part F: client cars ---")
_rng = [(350_000, 750_000), (800_000, 1_500_000), (300_000, 700_000), (1_500_000, 4_000_000), (30_000, 150_000)]
check("F  5 client predictions", lambda: len(client_predictions) == 5, "Clean the 5 client cars with YOUR clean function, then pipe.predict().")
check("F  client prices look realistic", lambda: all(lo <= p <= hi for p, (lo, hi) in zip(client_predictions, _rng)), "A price is far off. Did you clean client cars the same way? Are prices in rupees?")

print(f"\n===== SCORE: {_ok} / {_all} =====")

--- Part A-B: cleaning ---
PASS  A1 raw data loaded: 8128 rows, 13 columns
PASS  B  clean data has 6918 rows
PASS  B  no 'Test Drive Car' rows
PASS  B  km_driven outliers removed
PASS  B  mileage, engine, max_power are numbers
PASS  B  no 0 bhp cars
PASS  B  name, year, torque removed
PASS  B  car_age column exists (2020 - year)
PASS  B  19 brands (rare < 20 cars -> 'Other')
--- Part C: split + leakage ---
PASS  C  train 5534 rows, test 1384 rows
PASS  C  target is NOT in the features
PASS  C  empty cells NOT filled before split
PASS  C  text NOT encoded before split
--- Part D-E: pipeline + models ---
PASS  D  pipe is a sklearn Pipeline
PASS  D  pipe works on raw X_test
PASS  E  results table has 3+ models incl. Linear Regression
PASS  E  results has cv_r2, train_mae, test_mae, test_r2
PASS  E  best model test R2 >= 0.88
PASS  E  best model test MAE <= 90,000 rupees
--- Part F: client cars ---
PASS  F  5 client predictions
PASS  F  client prices look realistic

===== SCORE: 21 / 21 ==

1. Random Forest won. Its test MAE was about ₹78,776, which is approximately 0.79 lakh.

2. The model shows some overfitting because its train MAE was about ₹29,959, while its test MAE was about ₹78,776. The large difference means it performed much better on training data than on new data.

3. We trained on log(price) because car prices were highly skewed. Log transformation reduces the effect of extremely expensive cars and helps the model make more balanced predictions.

4. We did not fill empty cells before the split because the median would use information from the test data. This mistake is called data leakage.

5. We used ordinal encoding for owner because the categories have a natural order: First Owner, Second Owner, Third Owner, and Fourth & Above Owner. We used one-hot encoding for fuel because Diesel, Petrol, CNG, and LPG do not have a natural ranking.